In [15]:
import json
import time

import faiss
import numpy as np
import chromadb
from sentence_transformers import SentenceTransformer


In [ ]:
with open("naruto_s1.json", "r", encoding="utf-8") as f:
    dataset = json.load(f)

documents = [doc["raw_text"] for doc in dataset]
ids = [str(doc["id"]) for doc in dataset]
metadata = [{"url": doc["url"]} for doc in dataset]

print(f"Loaded {len(documents)} documents")


In [76]:
files = ["naruto_s1_backup.json", "naruto_s2.json"]

documents = []
ids = []
metadata = []

for file in files:
    with open(file, "r", encoding="utf-8") as f:
        dataset = json.load(f)

    documents.extend([doc["raw_text"] for doc in dataset])
    ids.extend([str(doc["id"]) for doc in dataset])
    metadata.extend([{"url": doc["url"]} for doc in dataset])

In [77]:
print(type(ids[0]))
print(type(ids[374]))

<class 'str'>
<class 'str'>


In [78]:
model = SentenceTransformer("all-MiniLM-L6-v2")

document_embeddings = model.encode(
    documents,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True,
).astype("float32")

print("Embedding shape:", document_embeddings.shape)


Batches: 100%|██████████| 14/14 [00:05<00:00,  2.77it/s]

Embedding shape: (442, 384)


In [79]:
chroma_client = chromadb.PersistentClient(path="./chroma_db")

if "naruto_documents" in {
    collection.name for collection in chroma_client.list_collections()
}:
    chroma_client.delete_collection("naruto_documents")

chroma_collection = chroma_client.create_collection(
    name="naruto_documents",
    configuration={"hnsw": {"space": "cosine"}},
)

chroma_collection.add(
    ids=ids,
    embeddings=document_embeddings.tolist(),
    documents=documents,
    metadatas=metadata,
)

print(f"Stored {chroma_collection.count()} documents in ChromaDB")


Stored 442 documents in ChromaDB


In [80]:
faiss_index = faiss.IndexFlatIP(document_embeddings.shape[1])
faiss_index.add(document_embeddings)

print(f"Stored {faiss_index.ntotal} documents in FAISS")


Stored 442 documents in FAISS


In [81]:
def chroma_search(query, top_k=5):
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    result = chroma_collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )

    return [
        {
            "id": result["ids"][0][rank],
            "score": 1 - result["distances"][0][rank],
            "url": result["metadatas"][0][rank]["url"],
            "text": result["documents"][0][rank],
        }
        for rank in range(len(result["ids"][0]))
    ]


In [ ]:
def faiss_search(query, top_k=5):
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    scores, indices = faiss_index.search(query_embedding, top_k)

    return [
        {
            "id": ids[index],
            "score": float(score),
            "url": documents[index]["url"],
            "text": documents[index]["raw_text"],
        }
        for score, index in zip(scores[0], indices[0])
        if index >= 0
    ]


In [90]:
def print_results(name, results):
    print(f"\n{name}")
    for rank, result in enumerate(results, start=1):
        print(f"\n{rank}. Score: {result['score']:.4f}")
        print("URL:", result["url"])
        print("Text:", result["text"][:300])


In [91]:
print("FAISS vectors:", faiss_index.ntotal)
print("Dataset documents:", len(dataset))
print("IDs:", len(ids))

FAISS vectors: 442
Dataset documents: 222
IDs: 442


In [92]:
query = "Nine tail fox attack"

start = time.perf_counter()
chroma_results = chroma_search(query)
chroma_time = time.perf_counter() - start

start = time.perf_counter()
faiss_results = faiss_search(query)
faiss_time = time.perf_counter() - start

print_results("ChromaDB results", chroma_results)
print(f"ChromaDB search time: {chroma_time:.6f} seconds")

print_results("FAISS results", faiss_results)
print(f"FAISS search time: {faiss_time:.6f} seconds")


<class 'numpy.int64'>
<class 'numpy.float32'>


TypeError: string indices must be integers, not 'str'

In [11]:
queries = [
    "Naruto fails an exam",
    "Sasuke fights Gaara",
    "Naruto learns a new technique",
    "a ninja team completes a dangerous mission",
    "Naruto protects his friends",
    "Sakura trains to become stronger",
    "Kakashi teaches Team 7",
    "a ninja passes an important test",
    "Sasuke leaves the Hidden Leaf Village",
    "Gaara uses his sand powers",
    "Naruto battles a dangerous enemy",
    "the Chunin Exams begin",
    "a character learns a new jutsu",
    "Team 7 goes on a mission",
    "a village is attacked",
    "Naruto and Sasuke fight together",
    "a ninja is injured during battle",
    "the characters prepare for a tournament",
    "someone tries to protect their family",
    "a powerful enemy is defeated",
]

def benchmark_search(search_function, queries, top_k=5):
    # Warm up model and database before measuring repeated searches.
    search_function(queries[0], top_k=top_k)
    timings = []
    results = {}

    for query in queries:
        start = time.perf_counter()
        results[query] = search_function(query, top_k=top_k)
        timings.append(time.perf_counter() - start)

    return results, np.asarray(timings)

chroma_benchmark_results, chroma_timings = benchmark_search(chroma_search, queries)
faiss_benchmark_results, faiss_timings = benchmark_search(faiss_search, queries)

print("Average search time")
print(f"ChromaDB: {chroma_timings.mean():.6f} seconds")
print(f"FAISS:    {faiss_timings.mean():.6f} seconds")
print()
print("Median search time")
print(f"ChromaDB: {np.median(chroma_timings):.6f} seconds")
print(f"FAISS:    {np.median(faiss_timings):.6f} seconds")

for query in queries:
    chroma_ids = [result["id"] for result in chroma_benchmark_results[query]]
    faiss_ids = [result["id"] for result in faiss_benchmark_results[query]]
    overlap = len(set(chroma_ids) & set(faiss_ids))
    print(f"\\n{query}")
    print(f"Top-5 result overlap: {overlap}/5")
    print(f"Same ranking: {chroma_ids == faiss_ids}")


Average search time
ChromaDB: 0.040738 seconds
FAISS:    0.009858 seconds

Median search time
ChromaDB: 0.010355 seconds
FAISS:    0.009747 seconds
\nNaruto fails an exam
Top-5 result overlap: 5/5
Same ranking: True
\nSasuke fights Gaara
Top-5 result overlap: 5/5
Same ranking: True
\nNaruto learns a new technique
Top-5 result overlap: 5/5
Same ranking: True
\na ninja team completes a dangerous mission
Top-5 result overlap: 5/5
Same ranking: True
\nNaruto protects his friends
Top-5 result overlap: 5/5
Same ranking: True
\nSakura trains to become stronger
Top-5 result overlap: 5/5
Same ranking: True
\nKakashi teaches Team 7
Top-5 result overlap: 5/5
Same ranking: True
\na ninja passes an important test
Top-5 result overlap: 5/5
Same ranking: True
\nSasuke leaves the Hidden Leaf Village
Top-5 result overlap: 5/5
Same ranking: True
\nGaara uses his sand powers
Top-5 result overlap: 5/5
Same ranking: True
\nNaruto battles a dangerous enemy
Top-5 result overlap: 5/5
Same ranking: True
\nthe 